# Environment spaces

Il notebook mostra `action_space` e `observation_space` degli ambienti custom gym.



In [18]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "beam_optimization" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

REPO_ROOT = PROJECT_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from beam_optimization.env.base_beam_env import BaseBeamEnv
from beam_optimization.config.adige import BEAM_STATE_FEATURES, PARAMETERS, STAGE_MARKERS

## How the environments are structured

These are custom Gymnasium environments for optimizing an accelerator beam.
The agent observes the beam state, chooses machine parameter changes,
and gets a reward when the beam improves.

The shared logic lives in `BaseBeamEnv`:
- `observation_space`: what the agent sees (beam features)
- `action_space`: what the agent can modify (16 deltas on TraceWin parameters)
- `reset`: starts a new episode
- `step`: applies an action, runs the simulation, computes the reward
- reward: difference between the new score and the previous one

Two concrete environments are available:
- `SurrogateEnv`: neural surrogate model — fast, no TraceWin call needed
- `TraceWinEnv`: real TraceWin simulator — slower, but reflects real physics

Both environments expose the same interface: same action space, same observation space.
The only difference is the simulator underneath.

In [19]:
# Try to import Markdown display utilities for Jupyter notebooks.
# If not available (e.g. plain Python script), set them to None.
try:
    from IPython.display import Markdown, display
except ImportError:
    display = None
    Markdown = None


class SpaceOnlyEnv(BaseBeamEnv):
    """Lightweight env used only to inspect spaces without running simulators."""

    def reset(self, *args, **kwargs):
        raise NotImplementedError("This env only displays spaces.")

    def step(self, action):
        raise NotImplementedError("This env only displays spaces.")


def fmt(value):
    """Format a numeric value to 6 significant digits."""
    return f"{float(value):.6g}"


def markdown_table(headers, rows):
    """Build a Markdown table from a list of headers and rows."""
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
    return "\n".join(lines)


def show(markdown):
    """Render Markdown in Jupyter if available, otherwise print plain text."""
    if display is None:
        print(markdown)
    else:
        display(Markdown(markdown))


# descriptions for each observation mode.
obs_descriptions = {
    "full": "all stages: 12 x 9 = 108 values",
    "final": "final stage only: 9 values",
    "final_with_beam0": "initial stage + final stage: 2 x 9 = 18 values",
}

env_names = ["SurrogateEnv", "TraceWinEnv"]
obs_modes = ["full", "final", "final_with_beam0"]

# Build the summary table: one row per (env, obs_mode) combination 
summary_rows = []

# Create a reference env just to read the action space bounds later.
example_env = SpaceOnlyEnv(obs_mode="full", action_scale=1.0)

for env_name in env_names:
    for obs_mode in obs_modes:
        env = SpaceOnlyEnv(obs_mode=obs_mode, action_scale=1.0)
        summary_rows.append(
            [
                env_name,
                f"`{obs_mode}`",
                obs_descriptions[obs_mode],
                env.observation_space.shape,
                env.observation_space.dtype,
                env.action_space.shape,
                env.action_space.dtype,
            ]
        )

# Build the feature table: one row per beam-state variable 
feature_rows = [[idx, f"`{name}`"] for idx, name in enumerate(BEAM_STATE_FEATURES)]

# Build the action table: one row per TraceWin parameter 
low = example_env.action_space.low
high = example_env.action_space.high

action_rows = []
for idx, param in enumerate(PARAMETERS):
    action_rows.append(
        [
            idx,
            f"`{param.name}`",
            f"`{param.key}`",
            param.marker,
            fmt(param.default),
            fmt(low[idx]),
            fmt(high[idx]),
        ]
    )

# Assemble and display the full report 
report = "\n\n".join(
    [
        "## Observation space and action space",
        markdown_table(
            ["Environment", "obs_mode", "Contents", "Obs shape", "Obs dtype", "Action shape", "Action dtype"],
            summary_rows,
        ),
        "## Features inside a single beam observation",
        markdown_table(["Index", "Feature"], feature_rows),
        "## Action space: 16 deltas on TraceWin parameters",
        "Each action is a vector of 16 values. Each component shifts one TraceWin parameter within the bounds shown below.",
        markdown_table(
            ["Index", "Parameter", "TraceWin key", "Marker", "Default", "Action min", "Action max"],
            action_rows,
        ),
        "Note: `SurrogateEnv` and `TraceWinEnv` share the same spaces; the only difference is which simulator runs underneath.",
    ]
)

show(report)

## Observation space and action space

| Environment | obs_mode | Contents | Obs shape | Obs dtype | Action shape | Action dtype |
| --- | --- | --- | --- | --- | --- | --- |
| SurrogateEnv | `full` | all stages: 12 x 9 = 108 values | (108,) | float32 | (16,) | float32 |
| SurrogateEnv | `final` | final stage only: 9 values | (9,) | float32 | (16,) | float32 |
| SurrogateEnv | `final_with_beam0` | initial stage + final stage: 2 x 9 = 18 values | (18,) | float32 | (16,) | float32 |
| TraceWinEnv | `full` | all stages: 12 x 9 = 108 values | (108,) | float32 | (16,) | float32 |
| TraceWinEnv | `final` | final stage only: 9 values | (9,) | float32 | (16,) | float32 |
| TraceWinEnv | `final_with_beam0` | initial stage + final stage: 2 x 9 = 18 values | (18,) | float32 | (16,) | float32 |

## Features inside a single beam observation

| Index | Feature |
| --- | --- |
| 0 | `npart_ratio` |
| 1 | `x0` |
| 2 | `y0` |
| 3 | `SizeX` |
| 4 | `SizeY` |
| 5 | `ex` |
| 6 | `ey` |
| 7 | `x'0` |
| 8 | `y'0` |

## Action space: 16 deltas on TraceWin parameters

Each action is a vector of 16 values. Each component shifts one TraceWin parameter within the bounds shown below.

| Index | Parameter | TraceWin key | Marker | Default | Action min | Action max |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | `AD.SO.01` | `ele[2][5]` | 2 | 0.365663 | -0.00314649 | 0.00314649 |
| 1 | `AD.SO.02` | `ele[4][5]` | 4 | 0.168963 | -0.00513743 | 0.00513743 |
| 2 | `AD.ST.04.X` | `ele[10][1]` | 10 | 0 | -3.03857e-05 | 3.03857e-05 |
| 3 | `AD.ST.04.Y` | `ele[10][2]` | 10 | 0 | -5.06879e-05 | 5.06879e-05 |
| 4 | `AD.1EQ.01` | `ele[12][2]` | 12 | -145.835 | -65.8423 | 65.8423 |
| 5 | `AD.1EQ.02` | `ele[16][2]` | 16 | -106.29 | -67.2851 | 67.2851 |
| 6 | `AD.D.02` | `ele[18][5]` | 18 | -0.0461848 | -3.28754e-05 | 3.28754e-05 |
| 7 | `AD.EM.6` | `ele[21][6]` | 24 | -213.9 | -191.897 | 191.897 |
| 8 | `AD.EM.8` | `ele[22][6]` | 24 | -16.8 | -849.527 | 849.527 |
| 9 | `AD.EM.10` | `ele[23][6]` | 24 | -1.67 | -1781.6 | 1781.6 |
| 10 | `AD.EM.12` | `ele[24][6]` | 24 | -86.9 | -5919.33 | 5919.33 |
| 11 | `AD.D.03` | `ele[26][5]` | 26 | 0.0461848 | -2.15583e-05 | 2.15583e-05 |
| 12 | `AD.1EQ.03` | `ele[30][2]` | 30 | -192.554 | -48.4793 | 48.4793 |
| 13 | `AD.1EQ.04` | `ele[35][2]` | 35 | 0 | -114.049 | 114.049 |
| 14 | `AD.ST.05.X` | `ele[38][1]` | 38 | 0 | -8.7603e-05 | 8.7603e-05 |
| 15 | `AD.ST.05.Y` | `ele[38][2]` | 38 | 0 | -7.929e-05 | 7.929e-05 |

Note: `SurrogateEnv` and `TraceWinEnv` share the same spaces; the only difference is which simulator runs underneath.